In [1]:
import polars as pl
from tcrtrifold.tcrdock_utils import (
    dgeom_ndarr_from_dgeom_series,
    mn_distr_from_dgeom_ndarr,
    mn_distance_from,
    un_cossin_embed,
)
from tcrtrifold.utils import filter_to_cog_thresh, FORMAT_ANTIGEN_COLS, FORMAT_TCR_COLS


iedb_II_conf = (
    pl.read_parquet("../../data/iedb_II/triad/iedb_II_triad.conf_af3.parquet")
    .drop(["receptor_id", "references"])
    .join(
        pl.read_parquet(
            "../../data/iedb_II/triad/iedb_II_triad.receptor_reference.parquet"
        ).select("job_name", "receptor_id", "references"),
        on="job_name",
        how="left",
    )
)
iedb_II_tcrdock = pl.read_parquet(
    "../../data/iedb_II/triad/staged/iedb_II_triad.af3_tcrdock.parquet"
)

iedb_II = iedb_II_conf.join(
    iedb_II_tcrdock.select("pred_dgeom_4", "job_name").with_columns(
        pl.col("pred_dgeom_4").struct.unnest()
    ),
    on="job_name",
    how="inner",
)

template_dgeom = pl.read_csv(
    "../../data/pdb/raw/ternary_templates_v2.tsv", separator="\t"
).with_columns(
    pl.struct(
        **{
            k: pl.col(k)
            for k in [
                "d",
                "torsion",
                "mhc_unit_x_is_negative",
                "tcr_unit_y",
                "tcr_unit_z",
                "tcr_unit_x_is_negative",
                "mhc_unit_y",
                "mhc_unit_z",
            ]
        }
    ).alias("dgeom"),
    pl.when(pl.col("mhc_class") == 1)
    .then(pl.lit("I"))
    .otherwise(pl.lit("II"))
    .alias("mhc_class"),
)

class_II_t_dgeom = dgeom_ndarr_from_dgeom_series(
    template_dgeom.filter(pl.col("mhc_class") == "II").select("dgeom").to_series()
)

class_II_distr = mn_distr_from_dgeom_ndarr(class_II_t_dgeom)

_, iedb_II_p_dgeom = mn_distance_from(
    dgeom_ndarr_from_dgeom_series(iedb_II.select("pred_dgeom_4").to_series()),
    *class_II_distr,
)

iedb_II = iedb_II.with_columns(pl.Series(name="p_dgeom", values=iedb_II_p_dgeom))

iedb_II = iedb_II.explode("receptor_id", "references")


iedb_I_conf = (
    pl.read_parquet("../../data/iedb_I/triad/iedb_I_triad.conf_af3.parquet")
    .drop(["receptor_id", "references"])
    .join(
        pl.read_parquet(
            "../../data/iedb_I/triad/iedb_I_triad.receptor_reference.parquet"
        ).select("job_name", "receptor_id", "references"),
        on="job_name",
        how="left",
    )
)
iedb_I = iedb_I_conf
iedb_I_tcrdock = pl.read_parquet(
    "../../data/iedb_I/triad/staged/iedb_I_triad.af3_tcrdock.parquet"
)

iedb_I = iedb_I_conf.join(
    iedb_I_tcrdock.select("pred_dgeom_4", "job_name").with_columns(
        pl.col("pred_dgeom_4").struct.unnest()
    ),
    on="job_name",
    how="inner",
)

class_I_t_dgeom = dgeom_ndarr_from_dgeom_series(
    template_dgeom.filter(pl.col("mhc_class") == "I").select("dgeom").to_series()
)

class_I_distr = mn_distr_from_dgeom_ndarr(class_I_t_dgeom)

_, iedb_I_p_dgeom = mn_distance_from(
    dgeom_ndarr_from_dgeom_series(iedb_I.select("pred_dgeom_4").to_series()),
    *class_I_distr,
)

iedb_I = iedb_I.with_columns(pl.Series(name="p_dgeom", values=iedb_I_p_dgeom))

iedb_I = iedb_I.explode("receptor_id", "references")

# REMOVEME TO ADD BACK IN
iedb_II_annot = pl.read_parquet(
    "../../data/iedb_II_full/triad/iedb_II_triad.annotated.parquet"
).filter(pl.col("triad_in_pdb"))

iedb_II = iedb_II.join(iedb_II_annot, on="job_name", how="anti")

# REMOVEME TO ADD BACK IN
iedb_I_annot = pl.read_parquet(
    "../../data/iedb_I_full/triad/iedb_I_triad.annotated.parquet"
).filter(pl.col("triad_in_pdb"))

iedb_I = iedb_I.join(iedb_I_annot, on="job_name", how="anti")

# additionally filter out
iedb_I = iedb_I.filter(pl.col("job_name") != "121f50b609621ebe92777820dad90eb9")

# docking_feats = [
#     "d",
#     "mhc_unit_y",
#     "mhc_unit_z",
#     "tcr_unit_y",
#     "tcr_unit_z",
#     "torsion",
#     "p_dgeom",
# ]

# interface_feats = [
#     "mean_p_tcr_pae",
#     "mean_tcr_p_pae",
#     "mean_mhc_tcr_pae",
#     "mean_tcr_mhc_pae",
#     "mean_p_tcr_contact_prob",
#     "mean_tcr_p_contact_prob",
#     "mean_mhc_tcr_contact_prob",
#     "mean_tcr_mhc_contact_prob",
#     "mean_p_tcr_interface_pae",
#     "mean_tcr_p_interface_pae",
#     "mean_tcr_pmhc_interface_pae",
#     "mean_pmhc_tcr_interface_pae",
#     "mean_p_tcr_interface_contact_prob",
#     "mean_tcr_p_interface_contact_prob",
#     "mean_tcr_pmhc_interface_contact_prob",
#     "mean_pmhc_tcr_interface_contact_prob",
#     "mean_p_mhc_pae",
#     "mean_mhc_p_pae",
#     "mean_mhc_p_interface_pae",
#     "mean_p_mhc_interface_pae",
#     # "mean_mhc_p_contact_prob",
#     # "mean_p_mhc_contact_prob",
#     "min_p_tcr_pae",
#     "min_mhc_tcr_pae",
#     "min_tcr_p_pae",
#     "min_tcr_mhc_pae",
#     "tcr_mhc_contacts",
#     "tcr_p_contacts",
# ]


# local_feats_II = [
#     "peptide_mean_pLDDT",
#     "peptide_mean_pLDDT_II",
#     "tcr_1_cdr_1_mean_pLDDT",
#     "tcr_1_cdr_2_mean_pLDDT",
#     "tcr_1_cdr_2_5_mean_pLDDT",
#     "tcr_1_cdr_3_mean_pLDDT",
#     "tcr_2_cdr_1_mean_pLDDT",
#     "tcr_2_cdr_2_mean_pLDDT",
#     "tcr_2_cdr_2_5_mean_pLDDT",
#     "tcr_2_cdr_3_mean_pLDDT",
#     "tcr_cdrs_mean_pLDDT",
#     "mhc_helices_mean_pLDDT",
# ]


# local_feats_I = [
#     "peptide_mean_pLDDT",
#     "tcr_1_cdr_1_mean_pLDDT",
#     "tcr_1_cdr_2_mean_pLDDT",
#     "tcr_1_cdr_2_5_mean_pLDDT",
#     "tcr_1_cdr_3_mean_pLDDT",
#     "tcr_2_cdr_1_mean_pLDDT",
#     "tcr_2_cdr_2_mean_pLDDT",
#     "tcr_2_cdr_2_5_mean_pLDDT",
#     "tcr_2_cdr_3_mean_pLDDT",
#     "tcr_cdrs_mean_pLDDT",
#     "mhc_helices_mean_pLDDT",
# ]

# summary_feats = [
#     "iptm",
#     "ptm",
#     "ranking_score",
# ]

# featnames_II = docking_feats + interface_feats + local_feats_II + summary_feats
# feat_type_II = (
#     ["docking"] * len(docking_feats)
#     + ["interface"] * len(interface_feats)
#     + ["local"] * len(local_feats_II)
#     + ["summary"] * len(summary_feats)
# )
# featnames_I = docking_feats + interface_feats + local_feats_I + summary_feats
# feat_type_I = (
#     ["docking"] * len(docking_feats)
#     + ["interface"] * len(interface_feats)
#     + ["local"] * len(local_feats_I)
#     + ["summary"] * len(summary_feats)
# )

pdb_v = pl.read_parquet("../../data/pdb/triad/pdb_validation_triad.conf_af3.parquet")
cresta = pl.read_parquet("../../data/cresta/triad/cresta_triad.conf_af3.parquet")

In [3]:
from tcrtrifold.utils import FORMAT_COLS

focal_cols = FORMAT_COLS + ["mean_p_tcr_interface_pae"]

all_dat = pl.concat(
    [
        iedb_I.with_columns(pl.lit("iedb").alias("source")).select(focal_cols),
        iedb_II.with_columns(pl.lit("iedb").alias("source")).select(focal_cols),
        pdb_v.with_columns(pl.lit("pdb").alias("source")).select(focal_cols),
        cresta.with_columns(pl.lit("cresta").alias("source")).select(focal_cols),
    ],
    how="align",
)

In [11]:
all_dat.write_csv("../../data/iedb_meta/supp_table_2.tsv", separator="\t")

## Supp table 1


In [5]:
pdb_af3 = pl.read_parquet(
    "../../data/pdb/triad/staged/pdb_triad.af3_rmsd.parquet"
).filter(pl.col("pdb") != "8trr")

In [12]:
pdb_af3.select(FORMAT_COLS + ["pdb"]).write_csv(
    "../../data/iedb_meta/supp_table_1.tsv", separator="\t"
)

In [ ]:
from tcrtrifold.utils import FORMAT_COLS, TCRDIST_COLS

all_dat_fmt = (
    all_dat.explode("references")
    .explode("receptor_id")
    .select(FORMAT_COLS + TCRDIST_COLS + featnames_II)
    .sort(by=["mhc_class", "cognate", "peptide"], descending=True)
)

all_dat_fmt.write_csv("../../data/iedb_meta/supp_table_1.tsv", separator="\t")